# Process SHERPA files to align lat/lon coords and calculate concentrations in future scenario-year

BCconc_emepV45_cams80_SURF_ug_PM25_rh50: the 2022 starting point  

EU_#scenario#_#year#: containing delta concentrations from 2022 to the considered year (2040, 2060, 2080, 2100) and scenario (7 scenarios, with delta > 0 meaning a reduction of concentrations)  

To have the concentrations in a future scenario-year, you need to make the difference between "BCconc_emepV45_cams80_SURF_ug_PM25_rh50" and "EU_#scenario#_#year#"  

The base 2022 starting point has float64 lat/lon coords, whilst the delta concentraions have float32 and are slightly misaligned. This script will check the coords are the same and then assign the delta concentrations the coordinates of the base file. Then it will calculate the difference between the base and the delta to get total concentrations in the future scenario-year.

In [ ]:
import os
import numpy as np
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [17]:
def check_and_replace_coords(base, delta):
    # Check dtype of coords for each input (likely different)
    print("Base coord dtype", base.latitude.dtype, "Delta coord dtype", delta.latitude.dtype)

    # If allclose returns True, the values are the "same" and are just precision artifacts
    assert np.allclose(base.latitude.values, delta.latitude.values, atol=1e-3), "Latitude coords are not the same"
    # Assign base latitude coords to delta
    delta = delta.assign_coords(
        latitude=base.latitude
    )

    # If allclose returns True, the values are the "same" and are just precision artifacts
    assert np.allclose(base.longitude.values, delta.longitude.values, atol=1e-3), "Longitude coords are not the same"
    # Assign base longitude coords to delta
    delta = delta.assign_coords(
        longitude=base.longitude
    )

    return delta

In [18]:
def calculate_future_conc(base, delta):
    with xr.set_options(keep_attrs=True):
        conc = base - delta
    return conc

In [20]:
FILE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")

years = [2040, 2060, 2080, 2100]
scenarios = ["H", "HL", "L", "LN", "M", "ML", "VL"]

base_file = "BCconc_emepV45_cams80_SURF_ug_PM25_rh50.nc"
base_path = os.path.join(FILE_DIR, base_file)
base = xr.open_dataarray(base_path)

for scenario in scenarios:
    for year in years:
        delta_file = f"EU_{scenario}_{year}.nc"
        delta_path = os.path.join(FILE_DIR, delta_file)
        delta = xr.open_dataarray(delta_path)

        delta = check_and_replace_coords(base, delta)
        conc = calculate_future_conc(base, delta)

        conc_file = f"EU_concentration_{scenario}_{year}.nc"
        conc_path = os.path.join(SAVE_DIR, conc_file)
        print(f"Saving to {conc_path}")
        conc.to_netcdf(conc_path)

print("All processing complete.")

Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/SHERPA/processed/EU_concentration_H_2040.nc
Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/SHERPA/processed/EU_concentration_H_2060.nc
Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/SHERPA/processed/EU_concentration_H_2080.nc
Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/SHERPA/processed/EU_concentration_H_2100.nc
Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/SHERPA/processed/EU_concentration_HL_2040.nc
Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/SHERPA/processed/EU_concentration_HL_2060.nc
Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/SHERPA/processed/EU_concentration_HL_2080.nc
Base coord dtype float64 Delta coord dtype float32
Saving to /glade/work/awells/EU_pm/S